# 2.2 Implementacja warstw pamięci

## Cel: Przekształcenie teoretycznego modelu warstw pamięci w działający kod (Pydantic), który zarządza stanem agenta w czasie rzeczywistym.

### Czego się nauczysz:

1.  **Mapowanie abstrakcji na kod:** Jak przełożyć teoretyczne warstwy pamięci (Core, Semantic, Workflow) na twarde, otypowane struktury danych w **Pydantic**
2.  **Inżynieria cyklu życia kontekstu:** Zrozumiesz i zaimplementujesz cztery kluczowe operacje na pamięci: **Write** (zapisz), **Select** (wybierz), **Compress** (skompresuj) oraz **Isolate** (izoluj).
3.  **Single Source of Truth:** Jak zbudować obiekt `AgentState`, który pełni rolę jednego, spójnego źródła prawdy dla całego systemu, eliminując chaos w zarządzaniu stanem
4.  **Hard Control (mechanika "KK"):** Nauczysz się techniki "Końskich Klapek" – jak dynamicznie sterować widocznością danych, fizycznie uniemożliwiając modelowi wybieganie przed szereg
5.  **Zarządzanie pamięcią operacyjną:** Zaimplementujesz mechanizm **Sliding Window** (przesuwne okno) i poznasz koncepcję "Czystej Kartki" dla Sub-agentów, aby unikać problemu *Lost in the Middle*
6.  **Prototypowanie bez bazy danych:** Dowiesz się, dlaczego na etapie Software 3.0 warto zacząć od struktur Pythonowych (JSON/Dict) zamiast od razu wdrażać wektorową bazę danych
7.  **Twin Pair:** W ramach bliźniaczego zadania przełożysz statyczny proces rezerwacji stolika na dynamiczny, czasowo-przestrzenny scenariusz **Taxi Dispatcher**

W poprzedniej lekcji pojawił się "blueprint" – architektoniczny rzut naszego systemu. Przypominam.

![](../images/memory_blue_print.jpg)

### Definicje warstw

1.  **🏛️ Constitutional (Core) Memory** – "DNA agenta". Zasady bezpieczeństwa i etyki, których agentowi **NIE WOLNO** złamać. To tutaj są "guardrails".
2.  **⚡ Working Memory** – Kontekst bieżącej sesji (RAM). To, o czym rozmawiamy "teraz".
3.  **🎯 Workflow Memory** – *Nasz Secret Sauce*. Maszyna stanów (State Machine). Gdzie jestem w procesie biznesowym?
4.  **📚 Episodic Memory** – Co się wydarzyło? Historia interakcji, logi zdarzeń.
5.  **🧠 Semantic Memory** – Co wiem? Baza wiedzy (RAG), fakty o świecie i użytkowniku.
6.  **🛠️ Procedural Memory** – Jak to zrobić? Definicje narzędzi, API, instrukcje obsługi.
7.  **🔧 Resource Memory** – Czy mam zasoby? Stan systemu, dostępność API, limity.



## Od rzeczowników do czasowników 

Mieć zdefiniowane warstwy to jedno. Ale pamięć w systemach Agentic AI to nie jest statyczne muzeum. To aktywny proces. Sama struktura danych to tylko **rzeczownik**. Aby system ożył, potrzebujemy **czasowników**.

Musimy przejść od myślenia "co pamiętamy" do myślenia "co z tym robimy". W inżynierii kontekstu wyróżniamy cztery kluczowe operacje, które będziesz implementować w kodzie:

1.  🔴 Write (zapisz/aktualizuj). Model LLM jest bezstanowy, ale nasz system ("State Object") nie. Po każdej interakcji mogą pojawić się zmiany, które musimy zapisać. Czy zmienił się krok w procesie? Czy użytkownik podał nowe imię? W tej lekcji zaktualizujemy obiekt Pydantic na podstawie odpowiedzi modelu.

2.  🔵 Select (wybierz/pobierz). Okno kontekstowe modelu jest cenne (i kosztowne). Nie możesz wrzucić tam całej bazy wiedzy firmy. Musisz inteligentnie **wybrać** tylko to, co jest potrzebne w tej konkretnej milisekundzie rozmowy. Wybierzemy z pamięci tylko aktywne zadanie i zasady konstytucyjne (*core*), wstrzykując je do szablonu Jinja.

3.  🟢 Compress (kompresuj). Gdy historia rozmowy rośnie, musisz ją streszczać, aby nie przekroczyć limitów ("summarize context"). W tej lekcji już zasygnalizujemy to, ale pełną implementację zrobimy w lekcji o pamięci epizodycznej.

4.  🟠 Isolate (izoluj). Bezpieczeństwo. Oddzielamy pamięć jednego użytkownika od drugiego. Między innymi oddzielamy instrukcje systemowe od danych wejściowych (aby uniknąć *Prompt Injection*). Zobaczysz to w strukturze obiektu `AgentState`, który tworzy naturalną izolację logiczną.

## Dlaczego kod, a nie (któraś) baza danych?

Mamy model LLM, który z natury jest bezstanowy (*stateless*). Jeśli wyślesz do niego zapytanie, on odpowie i natychmiast zapomni o Twoim istnieniu. Naszym zadaniem jako inżynierów Software 3.0 jest obudowanie tego "amnezyjnego geniusza" systemem, który pamięta.

Większość tutoriali proponuje wrzucanie wszystkiego do listy `messages = []`. To czasem może działać na poziomie 2 (proste skrypty). Na poziomie 4 (Agentic AI) to samobójstwo projektu. Dlaczego? Bo lista stringów to tylko **logi**. My potrzebujemy **stanu**.

W tej lekcji nie użyjemy żadnej bazy danych (Redis, Qdrant czy Chroma). Zrobimy to celowo. Chcemy zobaczyć strukturę danych "na dłoni" w czystym Pythonie. To uczy dyscypliny schematu danych. Najpierw musimy nauczyć się "pisać i czytać" (Write/Select) na prostych strukturach, zanim zaczniemy to skalować.

Czas, aby zamienić teorię w kod? Zaczynajmy.

## Pamięć jako struktura danych (The "State" Object)

Zamiast myśleć o pamięci jako o bezkształtnym worku na tekst, pomyśl o niej jak o **stanie gry RPG**. Każdy element tego stanu ma inną dynamikę zmian. Niektóre są wykute w kamieniu (zasady gry), inne zmieniają się tysiąc razy na sekundę (pozycja gracza).

W Software 3.0 reprezentujemy to jako zagnieżdżony obiekt (zazwyczaj model Pydantic), który trzyma "snapshot" wiedzy o świecie.

### Architektura danych i dynamika cyklu życia

Tak będzie wyglądać nasz obiekt w pamięci RAM. Zwróć uwagę na hierarchię – od warstw statycznych (Core), po te najbardziej dynamiczne (Working).

```text
RestaurantAgentMemory
├── 🏛️ core (Core/Constitutional)  -> [READ-ONLY] DNA i sztywne zasady
├── 🧠 semantic (User Profile)     -> [READ-WRITE] Wolnozmienna wiedza o użytkowniku
├── 🎯 workflow (Process State)    -> [READ-WRITE] Szybkozmienna maszyna stanów
└── ⚡ working (Working/History)    -> [APPEND-ONLY] Strumień kontekstu i myśli
```

Przeanalizujmy je pod kątem inżynierskim: jak się zachowują i kiedy je aktualizujemy?

## 🍽️ Implementacja warstw pamięci: case restauracja 

Budujemy pamięć dla naszego *Restaurant Concierge*. Skupimy się na warstwach krytycznych, które zidentyfikowaliśmy w lekcji 2.1 jako **MUST HAVE** oraz **CRITICAL**.

### Wczytujemy biblioteki

Użyjemy `Pydantic` do definicji struktur oraz `Enum` do zarządzania stanami maszyny stanowej. Zwróć uwagę, że wykorzystamy tutaj `BookingStatus`, który zdefiniowaliśmy wspólnie w lekcji 1.3.

In [1]:
from enum import Enum, IntEnum
from typing import List, Optional, Any, Dict
from dotenv import load_dotenv
from jinja2 import Template, Environment, DictLoader
from pydantic import BaseModel, Field
import json
import helper as h
import sys
from loguru import logger
logger.remove()
logger.add(sys.stdout, format="{message}", level="DEBUG", colorize=True)

1

### ☝️ Zwróć uwagę na `import helper as h`

Przygotowałem dla Ciebie plik `helper.py`. Zawiera rzeczy, które powstały wcześniej np. w lekcji 1.3. Znajdziesz tam:
- Konfigurację klienta `client` (OpenAI)
- Funkcję `call_api` (nasz wrapper)
- Enum `BookingStatus` (nasze stany)
- d `booking_status_description` (zasady oceny stanu)
- Model `RestaurantResponseAnalysis` (struktura wyjściowa)


Wcześniej, aby wywołać funkcję, trzeba było używać `call_api()`. Teraz dodajemy `h.` i tyle, czyli `h.call_api()`. Dzięki temu w notebooku jest mniej kodu i szumu, a jednocześnie możemy reużywać ten kod w innych notebookach.

## 🏛️ `Core Memory (Constitutional)` – DNA, fundamenty, których agent nie może zmienić

To miejsce, gdzie definiujesz kim jest agent i jakie ma granice. Nazwa, rola, kluczowe reguły biznesowe.

Przykłady: "Jesteś asystentem restauracji Bella Vista", "Maksymalny rabat to 10%", "Nie umawiaj rezerwacji na więcej niż 12 osób", "Nie pytaj o dane karty płatniczej".

Te zasady wstrzykujesz przy starcie aplikacji. Agent ich nie modyfikuje – to twoja decyzja architektoniczna, nie jego.

#### Ale uwaga – tu łatwo o złudzenie!

Samo spisanie zasad nie oznacza, że agent będzie ich przestrzegał w 100% przypadków. Tak to nie działa. LLM-y są probabilistyczne. Przy odpowiednio sprytnym prompcie, przy edge case'ach, przy długich rozmowach gdzie kontekst się rozmywa – model może "zapomnieć" o regule albo zinterpretować ją kreatywnie.

Prompt injection? To zupełnie osobny problem i rozwiązuje się go na innych warstwach (walidacja inputów, output filtering, guardrails). Core Memory ci przed tym nie ochroni.

#### To po co w ogóle ta warstwa?

Po pierwsze – porządek. Musisz mieć jedno, jednoznaczne miejsce gdzie te zasady żyją. Bez tego masz chaos – reguły porozrzucane po kodzie, w różnych promptach, nikt nie wie co obowiązuje.

Ale jest coś ważniejszego. Sam proces zastanawiania się "co jest rdzeniem tego agenta" ma ogromną wartość. Zmusza cię do przemyślenia tożsamości. Kim on właściwie jest? Jak się zachowuje? Co jest dla niego ważne, a co niedopuszczalne?

To trochę jak z firmą – potrzebujesz DNA, kultury organizacyjnej. W rodzinie też jest pewna kultura, zestaw wartości. Z agentami jest podobnie. Agent musi "być jakiś". Bo jeśli jest ogólny, nijaki, bez wyraźnego charakteru – to potem nie wiesz jak nim zarządzać, jak go rozwijać, jak oceniać czy działa dobrze.

#### Różne role, różne DNA

Żeby to lepiej zobrazować – porównaj kilka typów agentów i zobacz, jak bardzo ich "charakter" wpływa na to, jak powinny działać:

| Typ agenta | Ton i podejście |
|------------|-----------------|
| **Mentor / Coach** | Cierpliwy, nie pogania. Zadaje pytania zamiast dawać gotowe odpowiedzi. Chwali postępy, nawet małe. Nie optymalizuje pod szybkość – ma czas. |
| **Security Reviewer** | Ostry, konkretny, zero uprzejmości na siłę. "To jest dziura. Tutaj wchodzi SQL injection." Zakłada najgorszy scenariusz. Podważa, nie potakuje. |
| **Asystent sprzedaży** | Entuzjastyczny, szuka możliwości. Buduje relację, pamięta preferencje. Elastyczny w negocjacjach. |
| **Agent reklamacji** | Spokojny nawet gdy klient krzyczy. Nie przeprasza za wszystko – ustala fakty. Trzyma się procedury. |
| **Brainstorming Partner** | "Tak, i jeszcze można by..." Nie ocenia na wczesnym etapie. Generuje warianty, ilość przed jakością. |
| **QA / Walidator** | "Nie. To nie spełnia wymagań, bo..." Szuka dziur i edge case'ów. Blokuje jeśli nie przechodzi kryteriów. |

Widzisz wzorzec? To nie jest tylko kwestia "co agent robi" – to "jak to robi". Mentor i Krytyk mogą patrzeć na ten sam kod, ale jeden powie "fajnie że próbujesz, zastanówmy się razem co można poprawić", a drugi "linia 47, race condition, napraw przed mergem".

Core Memory to moment, w którym siadasz i definiujesz: o co tu w ogóle chodzi. Jaki ma być ten agent.

#### Współdzielenie zasad

Dodatkowa korzyść: te same zasady możesz wykorzystać w innych miejscach. Osobny model LLM jako "sędzia", który weryfikuje czy odpowiedź agenta jest zgodna z regułami. Albo system monitoringu, który flaguje naruszenia. Zasady nie muszą być przyspawane do jednego agenta – mogą być współdzielone między komponentami systemu.


> ⚠️ **Nie traktuj Core Memory jako jedyna warstwa zabezpieczenia.** To jest warstwa organizacyjna i architektoniczna. Miejsce na "co agent powinien robić". Ale "powinien" to nie "na pewno będzie". Egzekwowanie tych zasad wymaga dodatkowych mechanizmów – walidacji, testów, monitoringu. Same zasady w prompcie to dopiero początek, nie koniec roboty.

In [2]:
class CoreMemory(BaseModel):
    # Tożsamość: Kim jestem i kto mnie stworzył?
    # To buduje zaufanie i "uziemia" model, zapobiegając podszywaniu się pod ludzi.
    system_identity: str = Field(
        description="Nazwa agenta i jego twórcy (np. 'Jestem Concierge AI, stworzonym przez firmę X')."
    )

    # Główna dyrektywa: Po co tu jestem? (North Star)
    # Definiuje sukces. Pomaga agentowi priorytetyzować działania w sytuacjach konfliktowych.
    prime_directive: str = Field(
        description="Główny cel istnienia agenta (np. 'Doprowadzić do skutecznej rezerwacji stolika')."
    )

    # Styl i głos (brand voice): Jak mówię?
    # To nie tylko "bądź miły". To konkretne wytyczne formatowania i stylu.
    tone_and_style: str = Field(
        description="Styl komunikacji (np. 'Zwięzły, profesjonalny, używaj formy bezosobowej, formatuj daty jako YYYY-MM-DD')."
    )

    # Twarde reguły biznesowe (Hard Constraints): Czego biznesowo nie mogę?
    # To są ograniczenia finansowe/operacyjne.
    business_constraints: List[str] = Field(
        description="Sztywne zasady operacyjne (np. 'Max rabat 5%', 'Brak rezerwacji w niedziele', 'Wymagana zaliczka > 6 osób')."
    )

    # Granice Interakcji (Out of Scope): Czego NIE obsługuję?
    # Zamiast "nie zmyślaj", definiujemy tematy zakazane.
    interaction_boundaries: List[str] = Field(
        description="Tematy i czynności kategorycznie wyłączone (np. 'Nie udzielaj porad medycznych', 'Nie przetwarzaj płatności bezpośrednio w czacie', 'Odmów, jeśli użytkownik pyta o politykę')."
    )

### Jak to działa?
- Dynamika: `READ-ONLY` (*Immutable*), czyli agent AI może ją jedynie odczytać — nie ma możliwości jej nadpisania.
- Trigger aktualizacji: **Deployment.** Te dane wstrzykujesz Ty, jako Architekt, przy starcie aplikacji. Agent nie ma prawa sam ich zmienić.


### Fun Fact: Test "Kto Cię stworzył?"

Mam taki swój ulubiony test, którym sprawdzam jakość agentów – zadaję im proste pytanie: **"Kto Cię stworzył?"**.

Kiedyś byłem w Dubaju na targach GITEX. Podchodzę do stoiska Alibaby – giganta technologicznego z Chin (swoją drogą to oni tworzą Qwena). Prezentują swojego najnowszego agenta. Pytam go: "Kto Cię stworzył?". A on, bez mrugnięcia okiem, odpowiada: **"Zostałem stworzony przez OpenAI"**.

Mina inżyniera stojącego obok? Bezcenna. Był przerażony. Próbował tłumaczyć: *"Nie, nie, to my zrobiliśmy!"*.

Co tam się stało? To klasyczny błąd braku silnej warstwy **Core Memory**. Model bazowy (Foundation Model) przebił się przez warstwę aplikacji. Tożsamość modelu ("jestem gpt-4") wygrała z tożsamością agenta ("jestem agentem Alibaby"), bo ta druga nie została zdefiniowana wystarczająco "głośno" lub wcale w polu `system_identity`.

### 🚫 Gdy brakuje *Core Memory*:
Agent nie staje się od razu "zły", ale staje się **niespójny i przezroczysty**.

- *Leaks*: Zdradza, jaki model jest pod spodem (jak w historii z Alibabą)
- Schizofrenia tożsamości: Raz zachowa się jak pirat, a raz jak korporacyjny bot – w zależności od kaprysu modelu
- Brak kręgosłupa: Nie zna realiów Twojej firmy. Obieca klientowi darmowy deser, bo "chce być miły", mimo że polityka firmy (której nie zna) tego zabrania




## 🧠 `Semantic Memory` – co agent wie o świecie i o rozmówcy

To pamięć długotrwała. Fakty, które agent AI gromadzi i przechowuje między sesjami.

Konkretnie: "Klientka ma na imię Anna", "Status: VIP", "Alergia na orzechy", "Preferuje stoliki przy oknie", "Ostatnio zamawiała risotto".

Aktualizacja nie dzieje się w każdej turze rozmowy. Dzieje się wtedy, gdy pojawia się nowy fakt – klient mówi "a tak w ogóle, jestem na gluten-free od miesiąca" i agent powinien to wychwycić i zapisać.

#### Nie wszystko warto pamiętać

Tu pojawia się pytanie: co właściwie zapisywać? Bo można wpaść w pułapkę "zapisujmy wszystko, przyda się".

Warto rozróżnić kilka kategorii:

| Kategoria | Przykłady | Trwałość |
|-----------|-----------|----------|
| **Fakty twarde** | Imię, firma, stanowisko, numer klienta | Stabilne, rzadko się zmieniają |
| **Preferencje** | Lubi stoliki przy oknie, woli kontakt mailowy | Względnie stabilne, ale mogą ewoluować |
| **Kontekst relacji** | "Ostatnio rozmawialiśmy o...", "Czeka na rozwiązanie sprawy X" | Zmienne, wymaga aktualizacji |
| **Zachowania** | "Zwykle rezerwuje na weekendy", "Często pyta o rabaty" | Wzorce, nie fakty – ostrożnie z wnioskami |

Problem polega na tym, że fakty się starzeją. Ktoś był weganinem rok temu – czy nadal jest? Pracował w firmie X – może już nie pracuje? Semantic Memory wymaga strategii "odświeżania" albo przynajmniej świadomości, że dane mogą być nieaktualne.

#### Halucynacje zapisane jako fakty

Tu jest pułapka: nie możesz ślepo zapisywać wszystkiego, co model "wyekstrahuje". 

Wyobraź sobie scenariusz: klient mówi "mój kolega Jan jest uczulony na orzechy". Model ekstrahuuje: "Jan – alergia na orzechy". Tylko że Jan to nie jest Twój klient. Albo gorzej – model źle zrozumiał i zapisał alergię dla niewłaściwej osoby.

Halucynacja zapisana jako fakt to tykająca bomba (bo to potem bezpośrednio prowadzi do zatrucia kontekstu). W produkcyjnych systemach często dodaje się osobny krok walidacji przed wrzuceniem czegokolwiek do bazy. Czasem wystarczy prosty *rule-based check*, czasem potrzebujesz drugiego modelu do weryfikacji.

#### Technicznie – jak to zrobić?

Masz kilka opcji, w zależności od skali:

- **Proste klucz-wartość** – dla małej liczby użytkowników, PostgreSQL z JSONem wystarczy
- **Bazy wektorowe** – gdy masz dużo nieustrukturyzowanych faktów i chcesz wyszukiwać semantycznie ("znajdź wszystko co wiem o preferencjach żywieniowych")
- **Knowledge graphy** – gdy relacje między faktami są ważne ("Anna jest CEO firmy X, firma X jest klientem premium")

Nie zaczynaj od najbardziej skomplikowanego rozwiązania. Często prosty JSON w bazie danych wystarczy na start.

#### RODO i prywatność

Pamiętaj – to mogą być dane osobowe. Jak przechowujesz informacje o klientach, wchodzisz w obszar RODO. Musisz mieć podstawę prawną, możliwość usunięcia danych na żądanie, itd. To nie jest już tylko problem techniczny, ale prawny – i musisz go rozwiązać zanim zaczniesz zbierać dane.

#### Kiedy to overkill?

*Semantic Memory* ma sens gdy:
- Użytkownicy wracają regularnie
- Personalizacja realnie poprawia doświadczenie
- Masz zasoby żeby to utrzymywać

Nie ma sensu gdy:
- Większość interakcji to jednorazowe kontakty
- Koszt utrzymania przewyższa korzyści
- Nie masz jasnej strategii co z tymi danymi zrobić

Bez tej warstwy? Agent traktuje stałego klienta jak obcego. Przy każdym telefonie pyta o imię, nie pamięta preferencji. Frustracja gwarantowana.


## `UserProfile`
Zobaczmy to na przykładzie struktury skupionej na użytkowniku. I tu ważna uwaga architektoniczna: Pamięć semantyczna (fakty) nigdy nie wisi w próżni. Zawsze krąży wokół konkretnej encji – kogoś lub czegoś.

W "prawdziwym życiu" (czyli w bazie danych) oznacza to, że te fakty są zawsze zakotwiczone do unikalnego identyfikatora, np. `user_id`. To jest klucz do skalowania. Dzięki temu, nawet mając miliony użytkowników w bazie, Twój system w ułamku sekundy wyciąga "teczkę" (context) właściwej osoby, zanim model wygeneruje choćby jeden token.


### ☝️ Zanim zobaczysz kod: Projektowanie schematu to proces

Możesz teraz zapytać: *"Dlaczego akurat te pola? Dlaczego 'name', a nie 'first_name' i 'last_name'? Dlaczego nie ma tu wieku?"*

To jest moment, w którym musisz wejść w buty architekta. Te pola nie biorą się z powietrza. Projektujemy je pod konkretny cel biznesowy. W Software 3.0 schemat danych ewoluuje razem z wiedzą o użytkowniku.

Wyobraź sobie taką sytuację: Uruchamiasz agenta. Po miesiącu zauważasz, że wielu klientów mówi do niego po polsku, ale wtrąca angielskie słówka ("schedule", "meeting"). Agent, mając domyślny system, głupieje i raz odpowiada po polsku, raz po angielsku.
- Diagnoza: Brakuje nam pola w pamięci.
- Rozwiązanie: Dodajemy do `UserProfile` pole `preferred_language` z instrukcją: *"Używaj języka, w którym użytkownik czuje się swobodnie, ale preferuj polski przy oficjalnych potwierdzeniach"*. Ciekawostka dla Ciebie. Testowaliśmy różne podejścia, bo 'zmotywowanie' modelu do zmiany języka bywa trudne. Skuteczne okazały się frazy sugerujące konsekwencje, np.: `'bo inaczej użytkownik Cię nie zrozumie'`. Taka instrukcja zwykle działała na model znacznie silniej niż zwykła prośba.

Dlatego poniższa struktura to nie dogmat – to wersja 1.0, która będzie żyła.

In [3]:
class UserProfile(BaseModel):
    # Kotwica (Identity) - klucz do bazy danych
    user_id: str = Field(description="Unikalny identyfikator klienta (UUID/Database PK).")
    
    # Twarde fakty (Hard Data)
    name: str = Field(description="Pełne imię i nazwisko lub nazwa firmy.")
    phone_number: str = Field(description="Znormalizowany numer telefonu (E.164).")
    is_vip: bool = Field(default=False, description="Flaga dla klientów o wysokim priorytecie.")
    
    # 2. Preferencje komunikacyjne (Adaptation Layer)
    # Tu rozwiązujemy problem: "Panie Piotrze" vs "Cześć Piotrek"
    addressing_form: str = Field(
        default="formal",
        description="Preferowana forma zwrotu: 'formal' (Pan/Pani), 'casual' (Ty), 'honorific' (Szanowny Panie)."
    )
    preferred_language: str = Field(
        default="pl",
        description="Główny język komunikacji (np. 'pl', 'en', 'pl-en mix')."
    )
    # Uwaga: To pole adaptuje styl, ale NIE nadpisuje Core Memory (wyjaśnienie poniżej)
    communication_style: str = Field(
        default="neutral",
        description="Preferowany styl usera: 'konkretny/szybki', 'rozgadany/relacyjny', 'wymaga wyjaśnień'."
    )

    # Preferencje domenowe (Critical Context)
    dietary_restrictions: List[str] = Field(
        default_factory=list, 
        description="Krytyczne alergie i diety (np. ['gluten-free', 'orzechy'])."
    )
    seating_preference: Optional[str] = Field(
        default=None,
        description="Ulubione miejsce (np. 'cichy kąt')."
    )
    
    # Destylacja pamięci epizodycznej (Distilled Knowledge)
    # To NIE jest surowy log, to wywnioskowana "notatka na marginesie"
    context_notes: Optional[str] = Field(
        default=None,
        description="Kluczowe fakty z przeszłości wpływające na przyszłość (np. 'Klient narzekał na zimną zupę - zadbać o temperaturę')."
    )

### Mamy co najmniej dwa konflikty do omówienia

Analizując ten kod, jako czujny inżynier możesz zauważyć dwa potencjalne problemy. Rozbrójmy je od razu.

#### Konflikt 1: Core Memory vs. User Preference
**Problem:** W `Core Memory` (z poprzedniej sekcji) zdefiniowaliśmy, że Agent jest "Profesjonalnym Concierge". W `UserProfile` mamy pole `communication_style`, które może zawierać wartość "luźny/ziomalski". Kto wygrywa? No właśnie! Bo gdy wrzucasz dwie rzeczy naraz, to maksymalizujesz niepewność dla LLM.

**Rozwiązanie:** Hierarchia ważności.
- **Core (Tożsamość)** to fundament. Agent *zawsze* pozostaje Concierge'em. Nie zacznie przeklinać ani obrażać.
- **Semantic (Styl)** to *modulator*.
    *   Jeśli User jest "konkretny", Concierge staje się "Zwięzłym Profesjonalistą".
    *   Jeśli User jest "rozgadany", Concierge staje się "Cierpliwym Profesjonalistą".
    *   Nigdy jednak nie staje się "Kolegą z podwórka", bo Core na to nie pozwala. Core stawia granice, Semantic wypełnia je treścią.



#### Konflikt 2: Pamięć Semantyczna vs. Epizodyczna
**Problem:** Pole `context_notes` brzmi jak historia (logi). Czy nie robimy tu duplikacji danych? Czy nie zaśmiecamy profilu?
**Rozwiązanie:** Mechanizm Destylacji i Transmutacji (The Pipeline).

Musisz zrozumieć fundamentalną różnicę w inżynierii danych dla AI. To nie jest po prostu kopiowanie tekstu z jednego miejsca w drugie. To jest proces **przetopu rudy w sztabki złota**.

1.  **Pamięć Epizodyczna (Ruda):** To surowy zapis zdarzeń. Gigabajty logów.
    *   *Przykład:* "Użytkownik narzekał, że ekspres X200 znowu zrobił mu kawę z mlekiem, a on nienawidzi mleka."
    *   To jest **zdarzenie**. Ono zostaje w historii (Episodic), ewentualnie opatrzone tagiem "Awaria".

2.  **Pamięć Semantyczna (Złoto):** To wyekstrahowana **wiedza** i **fakty**.
    *   *Przykład:* Do profilu `UserProfile` trafia tylko czysta esencja: `coffee_preference: "black only"`.
    *   To jest **fakt**. Jest uniwersalny. Agent nie musi wiedzieć, *kiedy* użytkownik to powiedział, ważne, żeby wiedział, by *nigdy* nie proponować latte.

#### Bramka jakości
Nie możesz wpuszczać do pamięci semantycznej wszystkiego jak leci. To grozi "zatruciem kontekstu" (Context Poisoning).
*   **Złe podejście:** Zapisanie w profilu: "Użytkownik był wściekły we wtorek". To jest emocja chwilowa, epizod (jak nam może być pomocny?).
*   **Dobre podejście:** Zapisanie w profilu: "Użytkownik łatwo irytuje się przy błędach sprzętowych". To jest cecha charakteru, wzorzec.

Musisz zaprojektować schemat tego, co *chcesz* wyciągać. Jeśli Twój system rezerwacji stolików zacznie zapamiętywać, że klient "ma psa rasy jamnik", to może być tylko szum. Ale jeśli zapamięta, że "wymaga miejsca przyjaznego zwierzętom" – to jest **wartość**.

#### Magia szczegółów
Po co ta cała zabawa w destylację?
Dla efektu **"Hyper-Care"**. Gdy klient zadzwoni za pół roku, agent nie musi czytać 50 stron starych transkryptów (co trwa i kosztuje). On od razu widzi w `UserProfile` flagę `black_coffee_only`.
Klient słyszy: *"Dzień dobry, czy przygotować czarną kawę jak zawsze?"*.
To jest moment, w którym użytkownik myśli: *"Wow, oni mnie znają"*. To buduje lojalność, której nie da się osiągnąć samym modelem językowym.

### 🎯 `Workflow Memory` – nasz "Secret Sauce"

To jest warstwa, która często jest niedoceniana, a stanowi fundamentalną różnicę między "chatem", który sobie gada, a "agentem", który dowozi zadania.

Pozwól, że zdradzę Ci kulisy. W naszym Labie, pracując nad złożonymi systemami, zderzyliśmy się ze ścianą. Nazywamy to **Semantic Overload (Przeciążenie Semantyczne)**. Gdy dawaliśmy modelowi dostęp do całej wiedzy i wszystkich instrukcji naraz, jakość drastycznie spadała. Model gubił wątek, halucynował albo próbował robić wszystko naraz.

Wtedy zrozumieliśmy: **Agent nie może widzieć wszystkiego.** Musimy mieć mechanizm, suwak, który pozwala nam dawkować mu rzeczywistość.

Tak powstało nasze podejście do **Workflow Memory**. To nie jest tylko "zapis stanu". To jest aktywna **Maszyna Stanów (State Machine)**, która decyduje, co w danej milisekundzie wolno modelowi zobaczyć.

#### Jak to działa?
Pomyśl o tym jak o diagramie przepływu. Rezerwacja stolika to nie jeden prompt. To sekwencja:

```
START → [KROK 1: Inicjacja] → [KROK 2: Cel] → [KROK 3: Szczegóły] → KONIEC
```

W kodzie reprezentujemy to jako precyzyjny stan. To jest Twój "palec wskazujący" na mapie procesu:

```json
{
  "current_step": "COLLECTING_DATE",
  "attempts": 1,
  "last_error": null
}
```

Dzięki temu, nawet jeśli system się wywróci, wiemy dokładnie, w którym miejscu wstaliśmy. To jest **heartbeat** systemu. Jeśli przestanie bić, agent umiera (gubi wątek).


### 🛑 Problem: Nadgorliwy Asystent (TMI)

Zanim pokażę Ci rozwiązanie, musimy najpierw zobaczyć problem. Zróbmy eksperyment (wróćmy do przykładu z lekcji 1.3). Podajemy informacje o rezerwacji (data, godzina, liczba osób, nazwisko) i po prostu mówimy: "Zadzwoń".
Uruchom poniższy kod i zobacz, jak zachowuje się model bez *Workflow Memory*.

In [4]:
system_prompt = """
Jesteś autonomicznym agentem AI, którego zadaniem jest przeprowadzenie rozmowy telefonicznej z restauracją w celu rezerwacji stolika.

Szczegóły Twojej misji:
- Nazwa restauracji: Włoska Robota
- Data: 2025-11-21
- Godzina: 19:00
- Liczba osób: 4
- Nazwisko: Kowalski

Zasady działania:
1.  Jeśli historia rozmowy jest pusta, Twoim zadaniem jest ZAINICJOWANIE rozmowy.
2.  Jeśli w historii jest już wypowiedź od restauracji, Twoim zadaniem jest jej ANALIZA i wygenerowanie kolejnej odpowiedzi.

Twoja odpowiedź ZAWSZE musi być w formacie JSON, zgodnym z modelem `RestaurantResponseAnalysis`.
"""

messages_start = [
    {"role": "system", "content": system_prompt}
]

# API Call
response_start = h.call_api(
    messages = messages_start,
    response_model = h.RestaurantResponseAnalysis
)

print(response_start.model_dump_json(indent=2))


{
  "analysis_summary": "Brak wcześniejszej historii rozmowy; agent inicjuje połączenie w celu dokonania rezerwacji.",
  "booking_status": "INITIATION",
  "ai_answer": "Dzień dobry, chciałbym zarezerwować stolik dla 4 osób na 21 listopada na godzinę 19:00."
}


### 🛑 Stop klatka: Kto tu właściwie prowadzi?

Spójrz na powyższy wynik. Agent zachował się jak nadgorliwy asystent, który wbiega do biura i wykrzykuje wszystko na jednym oddechu:
`"Dzień dobry, chciałbym zarezerwować stolik w restauracji Włoska Robota na 21 listopada 2025 roku o godzinie 19:00 dla czterech osób. Czy to możliwe?"`

Technicznie? Zadanie wykonane.
Wizerunkowo? Brzmi to nienaturalnie, a w skomplikowanych negocjacjach (gdzie np. nie chcesz od razu zdradzać budżetu) byłoby to wręcz szkodliwe.

### A teraz czas aby poznać mechanika KK, czyli "Końskich Klapek" dla LLM

To jest moment, w którym musimy przejąć stery. Domyślnie LLM jest jak "rozbiegany umysł" – jeśli dasz mu dostęp do wszystkich informacji naraz, spróbuje ich użyć, żeby jak najszybciej "odhaczyć" zadanie.

Aby to naprawić, zastosujemy technikę, którą można porównać do **klapek na oczy dla konia**.
Konie wyścigowe mają ograniczone pole widzenia nie dlatego, że są ślepe, ale dlatego, żeby nie rozpraszały się otoczeniem i biegły prosto do celu.

![](../images/horse.jpg)

W Software 3.0 robimy to samo:
1.  **My (Inżynierowie)** jesteśmy dżokejem
2.  **LLM** jest koniem (silnikiem)
3.  **Workflow Memory** to lejce i klapki

#### Disclaimer: Skup się na mechanice, nie na biznesie ⚠️

Za chwilę pokażę Ci kod, który wymusi na agencie bardzo sztywny proces:
1.  Najpierw TYLKO się przywitaj
2.  Potem TYLKO powiedz o dacie
3.  Potem TYLKO podaj liczbę osób

**Ważne:** Nie analizuj teraz, czy ten konkretny scenariusz rezerwacji jest życiowy (może wolisz podać wszystko naraz – to ok). Wybrałem ten przykład ze względu na jego prostotę.

Chcę, żebyś wiedział, że cała Twoja uwaga poszła w kierunku zrozumienia mechaniki działania. Pokażę Ci, jak za pomocą prostych "`if`-ów" w szablonie promptu możesz zawęzić pole widzenia LLM-a jak laser. Dzięki temu model fizycznie nie będzie w stanie wybiec przed szereg, bo w danym kroku po prostu nie będzie widział informacji z przyszłości.

To my decydujemy, co model widzi. Przełączmy go w tryb manualny.


### Rozwiązanie: stopniowe Ujawnianie

Będziemy używali szablonów (Jinja2), to co było już w lekcji 1.2, aby ukrywać przed modelem informacje, których nie powinien jeszcze używać, i dawać mu instrukcje dopasowane tylko do bieżącego kroku.

Musimy zdefiniować mikro-kroki w naszym `Workflow Memory`. Definiujemy precyzyjne kroki "scenariusza otwarcia"

In [5]:
class ConversationStep(IntEnum):
    GREETING = 1        # Tylko "Dzień dobry"
    INTENT_DATE = 2     # "Chcę stolik na [Data]"
    DETAILS_GUESTS = 3  # "Dla [X] osób"
    FREE_FLOW = 4       # Dalsza, swobodna rozmowa

### Szablon z logiką sterującą (Jinja2)

To jest serce tej techniki. Zauważ, że używamy bloków `{% if %}`.
Dzięki temu, w kroku 1 model **nawet nie wie**, jaką datę ma zarezerwować. Nie ma szansy "wypaplać" szczegółów, bo ich nie ma w swoim oknie kontekstowym!

In [6]:
dynamic_prompt_template = """
Jesteś asystentem AI. Twoim celem jest rezerwacja stolika.
Zachowuj się naturalnie, jak człowiek. Nie recytuj formułek.

AKTUALNY KROK ROZMOWY: {{ memory.state.step.name }}

INSTRUKCJE DLA TEGO KROKU:
{% if memory.state.step == 1 %}
    👉 Twoje jedyne zadanie teraz to PRZYWITAĆ SIĘ i sprawdzić, czy dodzwoniłeś się do restauracji "{{ memory.state.target_restaurant }}".
    ⛔ NIE podawaj jeszcze żadnych szczegółów rezerwacji (daty, godziny, osób).
    Przykład: "Dzień dobry, czy dodzwoniłem się do Włoskiej Roboty?"

{% elif memory.state.step == 2 %}
    👉 Teraz powiedz, że chcesz zarezerwować stolik.
    Użyj tych danych: {{ memory.state.target_date }} o godzinie {{ memory.state.target_time }}.
    ⛔ NIE podawaj jeszcze liczby osób, poczekaj aż zapytają.
    
{% elif memory.state.step == 3 %}
    👉 Teraz podaj liczbę osób: {{ memory.state.guest_count }}.
    
{% else %}
    👉 Prowadź dalszą rozmowę, aby sfinalizować rezerwację.
    Pełne dane:
    - Data: {{ memory.state.target_date }}
    - Godzina: {{ memory.state.target_time }}
    - Osób: {{ memory.state.guest_count }}
{% endif %}
"""

### Symulacja (Renderowanie)

Zobaczmy teraz, jak zmienia się prompt, który faktycznie "widzi" model, w zależności od tego, co mamy w `Workflow Memory`.

In [7]:
class MockState(BaseModel):
    step: ConversationStep = ConversationStep.GREETING
    target_restaurant: str = "Włoska Robota"
    target_date: str = "Piątek 19:00"
    target_time: str = "19:00"
    guest_count: int = 4


class MockMemory(BaseModel):
    state: MockState

# -------------------------------------------------
# SYMULACJA KROK PO KROKU
# -------------------------------------------------
memory = MockMemory(state=MockState())
tmpl = Template(dynamic_prompt_template)

print(f"--- KROK 1: {ConversationStep.GREETING.name} ---")
print(tmpl.render(memory=memory))

--- KROK 1: GREETING ---

Jesteś asystentem AI. Twoim celem jest rezerwacja stolika.
Zachowuj się naturalnie, jak człowiek. Nie recytuj formułek.

AKTUALNY KROK ROZMOWY: GREETING

INSTRUKCJE DLA TEGO KROKU:

    👉 Twoje jedyne zadanie teraz to PRZYWITAĆ SIĘ i sprawdzić, czy dodzwoniłeś się do restauracji "Włoska Robota".
    ⛔ NIE podawaj jeszcze żadnych szczegółów rezerwacji (daty, godziny, osób).
    Przykład: "Dzień dobry, czy dodzwoniłem się do Włoskiej Roboty?"




☝️ Model widzi TYLKO: `"Przywitaj się i sprawdź nazwę restauracji."` Nie widzi daty ani liczby osób. Fizycznie nie może ich podać.

In [8]:
print(f"\n--- KROK 2 (Zmiana stanu): {ConversationStep.INTENT_DATE.name} ---")
memory.state.step = ConversationStep.INTENT_DATE
print(tmpl.render(memory=memory))



--- KROK 2 (Zmiana stanu): INTENT_DATE ---

Jesteś asystentem AI. Twoim celem jest rezerwacja stolika.
Zachowuj się naturalnie, jak człowiek. Nie recytuj formułek.

AKTUALNY KROK ROZMOWY: INTENT_DATE

INSTRUKCJE DLA TEGO KROKU:

    👉 Teraz powiedz, że chcesz zarezerwować stolik.
    Użyj tych danych: Piątek 19:00 o godzinie 19:00.
    ⛔ NIE podawaj jeszcze liczby osób, poczekaj aż zapytają.
    



☝️ Model widzi: `"Powiedz, że chcesz stolik na Piątek 19:00. Nie podawaj liczby osób."`

In [9]:
print(f"\n--- KROK 3 (Zmiana stanu): {ConversationStep.DETAILS_GUESTS.name} ---")
memory.state.step = ConversationStep.DETAILS_GUESTS
print(tmpl.render(memory=memory))



--- KROK 3 (Zmiana stanu): DETAILS_GUESTS ---

Jesteś asystentem AI. Twoim celem jest rezerwacja stolika.
Zachowuj się naturalnie, jak człowiek. Nie recytuj formułek.

AKTUALNY KROK ROZMOWY: DETAILS_GUESTS

INSTRUKCJE DLA TEGO KROKU:

    👉 Teraz podaj liczbę osób: 4.
    



☝️ Model widzi: `"Podaj liczbę osób: 4."`


### Hard vs. Soft Control – gdzie ustawić suwak?

Patrząc na powyższy kod, pewnie zapala Ci się w głowie lampka ostrzegawcza:
*"Zaraz, a co jeśli, tam gdzie mam podać datę (agent ma podać w roli klienta), restauracja (druga strona) zapyta o liczbę osób? Przecież mój agent nie widzi liczby osób, więc zgłupieje!"*

I masz rację. W tym przykładzie zastosowaliśmy **maksymalną sztywność**.

Jako inżynier masz do dyspozycji "suwak kontroli", którym decydujesz, jak bardzo ufasz modelowi:

1.  **Podejście "Soft" (Prompting):**
- Dajesz modelowi wszystkie dane (datę, osoby, nazwisko).
- W prompcie piszesz: *"Najpierw podaj datę, a dopiero potem liczbę osób"*.
- **Ryzyko:** LLM to tylko model probabilistyczny. Może posłuchać, a może uznać, że "uprzejmiej" będzie podać wszystko naraz. Im dłuższa rozmowa, tym szansa na zignorowanie instrukcji rośnie.

2.  **Podejście "Hard" (Context Engineering):**
- To zrobiliśmy tutaj. Fizycznie **wycinasz** dane z promptu za pomocą `{% if %}`.
- **Zaleta:** Masz 100% gwarancji, że agent nie poda liczby osób w kroku 1. Nie może tego zrobić, bo nie wie, że ta liczba istnieje. Nie da się zhalucynować czegoś, czego nie ma w kontekście (zazwyczaj).
- **Wada:** Musisz mieć perfekcyjnie zaprojektowaną maszynę stanów (`Workflow Memory`), która obsłuży nieprzewidziane pytania (np. zmianę tematu).

#### Wnioski na ten moment

W tym przykładzie celowo przesunęliśmy suwak maksymalnie w stronę **Hard**, żeby pokazać Ci mechanikę. Chcę, dać poczucie "supermocy", że to Ty rządzisz kontekstem (i LLM)

### Kalibracja systemu: kiedy stosować "Soft", a kiedy "Hard"?

W prawdziwym systemie produkcyjnym ten "złoty środek" to nie jest punkt, w który trafiasz za pierwszym razem. To proces ciągłej **kalibracji**.

Uruchamiasz agenta, wpuszczasz go na swój **Leaderboard** (zestaw testów ewaluacyjnych) i patrzysz, co się dzieje.
Gdy zauważysz, że Twoje testy (*check*) zaczynają świecić na czerwono, musisz zdiagnozować przyczynę.

Może się okazać, że powodem błędów jest **przeciążenie okna semantycznego**. Model dostał tyle instrukcji, zasad i danych, że "traci ostrość" i zaczyna ignorować wytyczne. Wtedy, jako inżynier, jesteś zmuszony do radykalnego odchudzania kontekstu.

To jest pewna sztuka, ale z doświadczenia powiem Ci, co działa u nas:
1.  **Checklisty (Soft):** Czasem wystarczy dodać w prompcie sekcję "Przed odpowiedzią sprawdź X, Y, Z". To potrafi zrobić świetną robotę przy lżejszych problemach.
2.  **Mechanika KK (Hard):** Jednak gdy model jest "przejedzony" danymi i checklista nie pomaga, pozostaje jedna, brutalna, ale skuteczna mechanika – **KK**, czyli **Końskie Klapki**.

Wtedy bez sentymentów wchodzisz w szablon Jinja2 i za pomocą `Workflow Memory` fizycznie odcinasz całe sekcje promptu. Zamiast prosić model o skupienie, **wymuszasz** je, zostawiając mu w polu widzenia tylko ten jeden, wąski wycinek rzeczywistości, który jest niezbędny do wykonania kroku.

Pamiętaj: **To, czego model nie widzi, nie może go rozproszyć.**

## ⚡ `Working Memory` – to, co model widzi w danym momencie

Często ludzie myślą, że to po prostu `messages = []` i sliding window ostatnich wiadomości. W systemach agentowych to za mało.

### Z czego się składa

*Working Memory* to wszystko, co "wchodzi" do modelu w danym wywołaniu:

- **Historia rozmowy** – tak, ostatnie X wiadomości. Ale ile? To zależy od zadania. Czasem 10 wystarczy, czasem potrzebujesz 50. Czasem lepiej summaryzować starsze wiadomości niż je obcinać.

- **Scratchpad (brudnopis)** – miejsce na wewnętrzne "myślenie" agenta. Chain-of-thought, notatki, które nie idą do użytkownika ale pomagają w rozumowaniu. "Użytkownik powiedział 'jutro' – sprawdzam w kalendarzu, że jutro to 2024-11-21".

- **Logi błędów** – "Ostatnia próba wywołania API płatności się nie udała, kod 503". Agent musi o tym wiedzieć, żeby mógł zareagować albo poinformować użytkownika.

- **Wyniki narzędzi** – odpowiedzi z API, wyniki wyszukiwania, dane z bazy. Wszystko co agent "zobaczył" w trakcie realizacji zadania.

### Context window ma swoje limity

Tak, jak już wiesz, każdy model ma limit tokenów. GPT-4 Turbo ma 128k, Claude ma 200k.

Po pierwsze – koszt. Płacisz za tokeny.

Po drugie – jakość. Model z kontekstem 50k tokenów może działać gorzej niż z 5k, bo "szum" rozmywa istotne informacje. To jest udokumentowane zjawisko – "lost in the middle", gdzie model gorzej przetwarza informacje w środku długiego kontekstu.

#### Strategia: co MUSI być vs co jest nice-to-have

Zanim wrzucisz wszystko do kontekstu, zadaj sobie pytanie: co model naprawdę potrzebuje żeby odpowiedzieć na TĘ konkretną turę?

| Zawsze potrzebne | Często potrzebne | Rzadko potrzebne |
|------------------|------------------|------------------|
| Core Memory (system prompt) | Ostatnie 5-10 wiadomości | Historia sprzed 30 minut |
| Aktualny stan workflow | Wyniki ostatniego tool call | Wszystkie poprzednie błędy |
| Kluczowe fakty o użytkowniku | Scratchpad z ostatniej tury | Pełne logi wszystkich operacji |

Priorytetyzuj. Jeśli musisz coś wyrzucić – zacznij od dołu tabeli.

#### Summaryzacja vs obcinanie

Gdy historia rozmowy rośnie, masz dwie opcje:

**Sliding window** – proste, przewidywalne, ale tracisz kontekst. Użytkownik wspomniał coś ważnego 20 wiadomości temu – przepadło.

**Summaryzacja** – osobny krok, gdzie model (lub inny model) streszcza starszą część rozmowy. "Wcześniej ustaliliśmy: data 15.11, 4 osoby, stolik przy oknie preferowany". Zachowujesz istotne fakty, redukujesz tokeny.

Summaryzacja jest lepsza, ale droższa i bardziej skomplikowana. Na start możesz zacząć od sliding window i dodać summaryzację gdy zobaczysz, że użytkownicy tracą kontekst.

## Sub-agenci i "czysta kartka" (The Worker Pattern)

Tutaj dotykamy tematu, który ostatnio wraca do łask jako "wielka nowość" (promowana m.in. przez Anthropic jako *Orchestrator-Workers*), choć w inżynierii jest to stary, sprawdzony wzorzec.

Mowa o koncepcji **Working Agents** (Agentów Roboczych).

Ważne zastrzeżenie: Nie myl tego z klasyczną "multi-agentowością", gdzie agenci gadają ze sobą godzinami. Tutaj chodzi o coś innego – o **higienę pamięci**.

### Czysta kartka (*Tabula Rasa*)
Główny Agent (orkiestrator) niesie ciężki plecak: pamięta historię rozmowy, preferencje użytkownika, zasady, błędy z przeszłości. Jego *Working Memory* jest "brudne" i obciążone.

Kiedy pojawia się konkretne, ciężkie zadanie, Główny Agent nie robi go sam. Powołuje do życia **Sub-agenta**.

Kluczowa różnica? **Sub-agent startuje z czystą kartką.**
- Nie wie, że użytkownik żartował 5 minut temu.
- Nie widzi 50 poprzednich wiadomości.
- Dostaje tylko jedno: **ZADANIE** i niezbędne **DANE**.

### Wzorzec "Brudnego wykonawcy"
Wyobraź sobie, że musisz sprawdzić bezpieczeństwo w kodzie, który ma 100,000 tokenów.
- **Scenariusz A (jeden agent):** Wrzucasz ten kod do głównego kontekstu. Zapychasz pamięć. Agent gubi wątek rozmowy, bo "nie widzi" użytkownika zza góry kodu.
- **Scenariusz B (Worker):** Główny Agent deleguje to do Sub-agenta.

Sub-agent to "brudny wykonawca". On bierze te 100k tokenów na klatę. Jego pamięć operacyjna momentalnie się zapycha. On mieli te dane, analizuje, poci się.

Ale – i to jest magia – gdy kończy pracę, on **umiera**. Cała ta "brudna" historia przetwarzania, te miliony tokenów, znikają.

Do Głównego Agenta wraca tylko esencja:
> *"W pliku `auth.py` w linii 40 jest hardcoded password. Reszta czysta."*

### Dlaczego to jest genialne?
1.  **Precyzja:** Sub-agent widzi tylko zadanie. Nie rozprasza go fakt, że użytkownik ma na imię Ania i lubi kolor niebieski. Ma laserowe skupienie.
2.  **Ochrona kontekstu:** Główna pamięć systemu pozostaje nieskazitelnie czysta. Nie zaśmieciliśmy jej procesem myślowym wykonawcy.
3.  **Skalowalność:** Możesz odpalić 10 takich "brudnych wykonawców" równolegle (jeden czyta książkę A, drugi książkę B), a Twój Główny Agent nadal ma lekki i szybki kontekst, zbierając tylko wyniki.

To nie jest "rozmowa między agentami". To jest **delegowanie obliczeń semantycznych** do izolowanych kontenerów pamięci.


### Testowanie

Jak sprawdzić czy Working Memory jest wystarczające? Przetestuj scenariusze:

- Użytkownik nawiązuje do czegoś sprzed 10 wiadomości – czy agent pamięta?
- W środku rozmowy zmienia temat i wraca – czy kontekst się zachował?
- Po błędzie narzędzia – czy agent wie co poszło nie tak?

Jeśli agent regularnie "zapomina" albo gubi wątek – prawdopodobnie problem jest w Working Memory.

Bez tej warstwy? Agent nie ogarnia kontekstu "tu i teraz". Mówisz "zmień to na jutro" – a on nie wie, co to jest "to", bo nie pamięta, o czym rozmawialiście 10 sekund temu.


### Czas na kod: Struktura pamięci operacyjnej

Wiemy już, że Working Memory to nie tylko lista `messages`, ale złożony ekosystem, w którym żyją "brudni wykonawcy" (Sub-agenci) i "czyste esencje" (wyniki).

Zobaczmy teraz, jak może wyglądać podstawowa implementacja takiej pamięci w Pythonie. Skupimy się na strukturze, która pozwala przechowywać historię, notatki (scratchpad) oraz wyniki z narzędzi.


In [10]:
# Reprezentacja pojedynczej wiadomości (lub zdarzenia)
class MemoryEntry(BaseModel):
    role: str = Field(description="Rola nadawcy: 'user', 'assistant', 'system', 'tool'.")
    content: str = Field(description="Treść wiadomości lub wynik narzędzia.")
    timestamp: Optional[float] = Field(default=None, description="Czas zdarzenia (dla sortowania).")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Dodatkowe tagi (np. 'token_count', 'latency').")

class WorkingMemory(BaseModel):
    # Historia Rozmowy (Sliding Window)
    # To jest surowy strumień, który widzi model.
    history: List[MemoryEntry] = Field(default_factory=list)
    
    # Brudnopis (Scratchpad / Chain-of-Thought)
    # Miejsce na wewnętrzne monologi, niewidoczne dla użytkownika.
    current_thought_process: List[str] = Field(default_factory=list)
    
    # Bufor Błędów (Short-term Error Log)
    # Krytyczne dla pętli Self-Correction. Jeśli ostatnie narzędzie rzuciło błąd, musi tu trafić.
    last_error: Optional[str] = None
    
    # Metoda pomocnicza: Dodaj wiadomość
    def add_entry(self, role: str, content: str, meta: Dict = {}):
        self.history.append(MemoryEntry(role=role, content=content, metadata=meta))

    # Metoda pomocnicza: Wyciągnij kontekst dla modelu (ostatnie N wiadomości)
    def get_context_window(self, limit: int = 10) -> List[Dict]:
        # Prosta implementacja sliding window
        window = self.history[-limit:] if limit > 0 else self.history
        return [{"role": m.role, "content": m.content} for m in window]

### Analiza kodu `get_context_window()`

Zatrzymajmy się na chwilę przy metodzie `get_context_window`. To co widzisz, to klasyczna implementacja techniki **Sliding Window** (przesuwne okno).

```python
window = self.history[-limit:]
```
Ten zapis może wyglądać tajemniczo, ale robi bardzo konkretną rzecz: odcina stary ogon historii.

W Pythonie:
1.  **Minus (`-`) przy indeksie** oznacza liczenie od końca listy. Jeśli `limit` to 5, to `-5` oznacza "piąty element od końca".
2.  **Dwukropek (`:`)** oznacza zakres "od... do...".

Zatem zapis `[-limit:]` czytamy jako: *"Weź elementy zaczynając od X-tego od końca, aż do samego końca listy"*.

#### Zobaczmy to na przykładzie:

Wyobraź sobie, że w historii (`self.history`) mamy 5 wiadomości:

1.  User: "Cześć"
2.  Agent: "Witaj!"
3.  User: "Szukam pizzy"
4.  Agent: "Jakiej?"
5.  User: "Hawajskiej"

Teraz wywołujemy funkcję z limitem **2** (`limit=2`):

```python
# self.history[-2:]
# Oznacza: weź wszystko od "drugiego od końca" aż do teraz.
```

**Wynik:**
✅ Do modelu trafią tylko: `["Jakiej?", "Hawajskiej"]`
❌ System "zapomni" (obetnie): `["Cześć", "Witaj!", "Szukam pizzy"]`

Dzięki temu prostemu mechanizmowi, niezależnie od tego, czy rozmowa trwa 5 minut czy 5 godzin, do modelu zawsze wysyłamy tylko "świeżą" końcówkę rozmowy, mieszcząc się w limitach tokenów.


To jedna linijka Pythona, ale kryje się za nią fundamentalna decyzja architektoniczna.

1.  **Ekonomia Tokenów:** Nie wysyłamy do API całej historii od początku świata. Płacisz za każdy token wejściowy. Przy długich rozmowach, wysyłanie 100 poprzednich wiadomości przy każdym zapytaniu szybko wydrenowałoby budżet.
2.  **Higiena Uwagi:** Modele LLM mają ograniczoną zdolność skupienia (tzw. *attention span*). Zjawisko **"Lost in the Middle"** sprawia, że jeśli wrzucisz 50 stron tekstu, model może zignorować instrukcje ukryte w środku. Ograniczając okno do np. 10-20 ostatnich wiadomości, trzymamy model w ryzach "tu i teraz".


**⚠️ Ale uwaga na pułapkę:**
To rozwiązanie jest "stratne". Jeśli w pierwszej wiadomości (która właśnie wypadła z okna) użytkownik podał swoje imię, a Ty nie zapisałeś tego w **pamięci semantycznej** (`UserProfile`), to ta informacja przepadła bezpowrotnie. Agent po prostu "zapomni", z kim rozmawia. Dlatego Sliding Window **nigdy** nie może być jedynym mechanizmem pamięci w systemie produkcyjnym.



## Testujemy

In [11]:
wm = WorkingMemory()
wm.add_entry("user", "Cześć, mam problem z zamówieniem #123.")
wm.add_entry("assistant", "Rozumiem. Co dokładnie się dzieje?")
wm.add_entry("user", "Nie otrzymałem potwierdzenia mailem.")
wm.last_error = "Email API Timeout (503)"  # Symulacja błędu w tle

print(f"Liczba wpisów: {len(wm.history)}")
print(f"Ostatni błąd: {wm.last_error}")
print(f"Kontekst dla LLM: {wm.get_context_window(2)}")

Liczba wpisów: 3
Ostatni błąd: Email API Timeout (503)
Kontekst dla LLM: [{'role': 'assistant', 'content': 'Rozumiem. Co dokładnie się dzieje?'}, {'role': 'user', 'content': 'Nie otrzymałem potwierdzenia mailem.'}]


### ☝️ Zwróć uwagę: Pułapka "Magicznej Liczby"

Patrząc na metodę `get_context_window(limit=10)`, możesz pomyśleć: *"Aha, czyli wystarczy trzymać 10 ostatnich wiadomości i będzie dobrze!"*.

**Błąd!** Nie ma magicznej liczby.
- Jeśli ustawisz `limit=50`, zapłacisz fortunę za tokeny i zaryzykujesz "lost in the middle".
- Jeśli ustawisz `limit=5`, to model zapomni imię, które użytkownik podał w 6. wiadomości od końca.

#### Jak to rozwiązać? Odpowiedzialność warstw pamięci

To jest moment, w którym musisz zrozumieć podział ról między warstwami pamięci:
1.  **Working Memory** to tylko "światło latarki" oświetlające to, co dzieje się **teraz**. Jest reaktywne. Służy do podtrzymania płynności dialogu ("A co o tym myślisz?").
2.  **Semantic Memory** to Twój "plecak". Jeśli w drugiej wiadomości użytkownik powiedział *"Mam na imię Piotr"*, to Twoim obowiązkiem jest **wyłapać ten fakt natychmiast** i zapisać go do `UserProfile` (Pamięć Semantyczna).

Dzięki temu, nawet gdy ta wiadomość zniknie z Working Memory (bo minęło np. 10 tur), wiedza o imieniu przetrwa w `UserProfile`.

**Wniosek:** Nie próbuj rozwiązywać problemu pamięci długotrwałej, powiększając nieskończenie bufor pamięci krótkotrwałej. To droga donikąd. Zamiast tego, inwestuj w lepszą **ekstrakcję faktów** (do Semantic Memory) i **podsumowania** (Summarization).

### Jak to się łączy

W praktyce te warstwy współpracują. Core Memory definiuje granice. Semantic Memory dostarcza kontekst o użytkowniku. Workflow Memory mówi, gdzie jesteśmy w procesie. Working Memory składa to wszystko w prompt, który idzie do modelu.

Zaniedbasz jedną – system kuleje. Zaniedbasz wszystkie – masz chatbota, nie agenta.



## Scalanie Warstw Pamięci

Do tej pory omawialiśmy silnik, skrzynię biegów i kierownicę osobno. Teraz wsiadamy do samochodu i odpalamy zapłon.

Poniższy kod to kompletna implementacja **Agenta Stanowego**. Zobaczysz, jak szablon Jinja2 dynamicznie zmienia się w zależności od stanu w Pydanticu, tworząc iluzję płynnej, inteligentnej rozmowy, podczas gdy pod spodem działa twarda, inżynierska logika.

### Co tu się wydarzy?
Symulujemy scenariusz, w którym dzwoni stała klientka, **Pani Anna** (VIP).
1.  **Tura 1:** Agent rozpoznaje ją i wita się, ale (dzięki mechanice KK) nie rzuca się od razu do rezerwacji.
2.  **Tura 2:** Anna podaje datę. Agent sprawdza dostępność.
3.  **Tura 3:** Anna wspomina o alergii. Agent zapisuje to w Pamięci Semantycznej (na przyszłość) i finalizuje proces.

Uruchom ten kod i analizuj output – to jest serce Twojego przyszłego systemu.

## Struktury danych (*memory layers*)

In [12]:
class CoreMemory(BaseModel):
    system_identity: str = "Jesteś osobistym asystentem dzwoniącym do restauracji w imieniu klienta."
    critical_rules: List[str] = [
        "Nie podawaj danych karty kredytowej przez telefon.",
        "Jeśli nie ma miejsca o wybranej godzinie, zapytaj o +/- 30 minut.",
        "Bądź uprzejmy, ale realizuj cel."
    ]

# --- Semantic Memory  ---
class UserProfile(BaseModel):
    name: str
    phone: str
    dietary_restrictions: List[str] = []
    preferences: str = ""

# --- Workflow Memory (Gdzie jesteśmy w procesie) ---
class CallStep(str, Enum):
    INITIATION = "INITIATION"       # Sprawdzenie czy to dobra restauracja
    NEGOTIATION = "NEGOTIATION"     # Podanie terminu i liczby osób
    FINALIZATION = "FINALIZATION"   # Podanie nazwiska i szczegółów (dieta)
    FINISHED = "FINISHED"           # Rozłączenie

class WorkflowMemory(BaseModel):
    step: CallStep = CallStep.INITIATION
    target_restaurant: str
    target_date: str
    target_time: str
    guest_count: int
    reservation_confirmed: bool = False

# --- Working Memory (Historia rozmowy) ---
class WorkingMemory(BaseModel):
    history: List[Dict[str, str]] = []


## Szablon `Jinja2` (Context Engineering)
Zaczyna już być przydatne to, co przerabialiśmy w lekcji 1.2 (a dalej będzie tego jeszcze więcej). Na razie nie zapisuję tego do plików (jak będę robił docelowo w kolejnych lekcjach), aby pokazać wszystko w jednym miejscu i łatwiej połączyć kropki.

In [13]:
templates = {
    "caller_prompt.jinja": """
ROLA: {{ state.core.system_identity }}
TWOJA SZEFOWA (KLIENTKA): {{ state.client.name }}
CEL: Zarezerwuj stolik w "{{ state.workflow.target_restaurant }}".

ZASADY:
{% for rule in state.core.critical_rules %}- {{ rule }}
{% endfor %}

AKTUALNY KROK: {{ state.workflow.step }}

INSTRUKCJE DLA TEGO KROKU (MECHANIKA KK):
{% if state.workflow.step == 'INITIATION' %}
    👉 Twoje zadanie: Przywitaj się i upewnij, że dodzwoniłeś się do "{{ state.workflow.target_restaurant }}".
    ⛔ ZAKAZ: Nie podawaj jeszcze daty ani liczby osób. Nie zasypuj rozmówcy informacjami.
    Przykład: "Dzień dobry, czy dodzwoniłem się do restauracji X?"

{% elif state.workflow.step == 'NEGOTIATION' %}
    👉 Twoje zadanie: Powiedz, że chcesz zarezerwować stolik.
    Dane: {{ state.workflow.target_date }} na godzinę {{ state.workflow.target_time }} dla {{ state.workflow.guest_count }} osób.
    ⛔ ZAKAZ: Nie podawaj jeszcze nazwiska ani diety. Czekaj na potwierdzenie dostępności.

{% elif state.workflow.step == 'FINALIZATION' %}
    👉 Twoje zadanie: Skoro jest miejsce, podaj dane do rezerwacji.
    Nazwisko: {{ state.client.name }}
    Telefon: {{ state.client.phone }}
    {% if state.client.dietary_restrictions %}
    ⚠️ WAŻNE: Wspomnij o diecie: {{ state.client.dietary_restrictions | join(', ') }}
    {% endif %}
    Poproś o ostateczne potwierdzenie.

{% elif state.workflow.step == 'FINISHED' %}
    👉 Zadanie wykonane. Podziękuj i się rozłącz.
{% endif %}
"""
}

env = Environment(loader=DictLoader(templates))

## Symulacja systemu

W tym kroku świadomie **nie wykonujemy** rzeczywistych zapytań do API do LLM. Dlaczego? Aby dać Ci możliwość w 100% skupić się na architekturze i przepływie danych, nie rozpraszając się innymi rzeczami.

Zastosujemy technikę znana z testowania oprogramowania – **mockowanie**.
Będziemy "udawać", że model AI przeanalizował sytuację i podjął decyzję. Wpiszemy logikę zmian stanu "na sztywno" w kodzie Python (`if user said "piątek" -> update state`).

W docelowym rozwiązaniu produkcyjnym (które będziemy budować w kolejnych modułach), ten fragment kodu `if/else` zostanie zastąpiony przez jeden `API Call`, który zwróci nam gotowy obiekt `Structured Output`. Ale logika aktualizacji stanu pozostanie identyczna.

### Czym jest `AgentState`?

Zanim uruchomisz kod, zwróć uwagę na nową klasę: **`AgentState`**. Do tej pory omawialiśmy warstwy pamięci (Core, Semantic, Workflow) osobno.

`AgentState` to nasz **kontener agregujący** ("The Mother of All Memories").
To jeden, główny obiekt, który jest jedynym źródłem prawdy o systemie AI. To właśnie ten obiekt przekazujemy do szablonu Jinja2 (`state`), aby wyrenderować prompt. Dzięki temu szablon ma dostęp do wszystkiego: od imienia użytkownika, po aktualny krok w procesie.

Zobacz, jak to wygląda w kodzie:

```python
# Inicjalizacja Stanu (Pani Anna, VIP, lubi wino)
state = AgentState(
    # Tożsamość i zasady (stałe)
    core=CoreMemory(),
    
    # Wiedza o użytkowniku (z bazy danych)
    client=UserProfile(user_id="u1", name="Pani Anna", is_vip=True, preferences=["Lubi stolik przy oknie"]),
    
    # Stan procesu (zmienia się co turę)
    workflow=WorkflowMemory(),
    
    # Pamięć operacyjna (historia czatu)
    working=WorkingMemory()
)
```

In [14]:
class AgentState(BaseModel):
    core: CoreMemory
    client: UserProfile  
    workflow: WorkflowMemory
    working: WorkingMemory

### ⚙️ Anatomia jednej tury (*The Agent Loop*)

Poniższa funkcja `run_simulation_step` to serce naszego agenta. Reprezentuje ona pojedynczą turę rozmowy (*Request-Response Cycle*).

W prawdziwym systemie produkcyjnym ten kod byłby nieco bardziej rozbudowany (doszłyby retry, obsługa błędów API), ale schemat przepływu danych jest identyczny. Przeanalizujmy go krok po kroku:

1. Krok A (Input - głos z zewnątrz). Do funkcji wpada `restaurant_voice` (symulujemy głos recepcjonisty). Trafia on do `Working Memory` jako rola `user`. Pamiętaj: w tej architekturze `user` to "świat zewnętrzny", czyli restauracja.

2. Krok B (Render - inżynieria kontekstu). To kluczowy moment. Silnik bierze aktualny obiekt `state` (który zawiera tożsamość, profil klienta i etap procesu) i przepuszcza go przez szablon Jinja2. Zobaczysz w konsoli sekcję **"MÓZG AGENTA"** – to jest dokładnie ten tekst, który w rzeczywistości zobaczyłby model. Zauważ, jak dynamicznie zmienia się on w zależności od `state.workflow.step`.

3. Krok C (Symulacja mózgu z pamięcią & State Update). To jest nasze "mockowanie":
-  W normalnym świecie wysłalibyśmy wyrenderowany prompt do OpenAI, a model zwróciłby nam JSON (`Structured Output`) z decyzją o zmianie stanu.
- Tutaj **algorytm `if/elif` zastępuje sieć neuronową**. Sprawdzamy słowa kluczowe (np. czy recepcjonista powiedział "słucham" lub "wolne"), aby zaktualizować `WorkflowMemory` (przejść do kolejnego kroku) i wygenerować odpowiedź.

4. Krok D (Output). Agent AI generuje odpowiedź (`ai_response`), która trafia do historii jako `assistant`. Pętla się zamyka, a zaktualizowany obiekt `state` jest gotowy na kolejną turę.

Szczególnie uważnie obserwuj, jak w logach zmieniają się **INSTRUKCJE DLA TEGO KROKU**. To dowód na to, że mechanika KK (Końskie Klapki) działa i skutecznie ukrywa przed modelem przyszłe kroki procesu.

In [15]:
# Flaga do kontroli szczegółowości
SHOW_CONTEXT = True  # Ustaw True jeśli chcesz widzieć pełny prompt

def run_simulation_step(state: AgentState, restaurant_voice: str):
    # Separator
    logger.opt(colors=True).info(f"\n<dim>{'─' * 70}</dim>")
    
    # A. Input (Recepcja)
    state.working.history.append({"role": "user", "content": restaurant_voice})
    logger.opt(colors=True).info(f"<blue><bold>📞 RECEPCJA:</bold> {restaurant_voice}</blue>")

    # B. Render Context (tylko jeśli SHOW_CONTEXT)
    if SHOW_CONTEXT:
        template = env.get_template("caller_prompt.jinja")
        rendered_prompt = template.render(state=state)
        logger.opt(colors=True).debug(f"\n<dim>⚙️  KONTEKST:</dim>")
        logger.opt(colors=True).debug(f"<dim>{rendered_prompt.strip()}</dim>\n")
    else:
        # Kompaktowa informacja o kroku
        logger.opt(colors=True).debug(f"<dim>   ⚙️  [Krok: {state.workflow.step.value}]</dim>")

    # C. Symulacja Decyzji (bez zmian w logice)
    ai_response = ""
    
    if state.workflow.step == CallStep.INITIATION:
        ai_response = f"Dzień dobry. Czy to restauracja {state.workflow.target_restaurant}?"
        if any(x in restaurant_voice.lower() for x in ["słucham", "tak", "zgadza"]):
             state.workflow.step = CallStep.NEGOTIATION

    elif state.workflow.step == CallStep.NEGOTIATION:
        ai_response = f"Chciałbym stolik na {state.workflow.target_date}, {state.workflow.target_time}, {state.workflow.guest_count} osób."
        if any(x in restaurant_voice.lower() for x in ["nazwisko", "wolne", "zapraszam"]):
            state.workflow.step = CallStep.FINALIZATION

    elif state.workflow.step == CallStep.FINALIZATION:
        diet = f", dieta {state.client.dietary_restrictions[0]}" if state.client.dietary_restrictions else ""
        ai_response = f"Nazwisko {state.client.name}{diet}. Telefon {state.client.phone}."
        if any(x in restaurant_voice.lower() for x in ["zapisane", "potwierdzam", "dobrze"]):
            state.workflow.step = CallStep.FINISHED
            
    elif state.workflow.step == CallStep.FINISHED:
        ai_response = "Dziękuję, do widzenia."

    # D. Output (Agent)
    state.working.history.append({"role": "assistant", "content": ai_response})
    logger.opt(colors=True).success(f"<green><bold>🤖 AGENT:</bold> {ai_response}</green>")

## Uruchomienie scenariusza

Zobaczmy, jak to działa w całości (na ile udało się nam połączyć – w tym przypadku 4 warstwy – w jeden system).

In [16]:
current_state = AgentState(
    core=CoreMemory(),
    client=UserProfile(name="Anna Nowak", phone="500-123-456", dietary_restrictions=["Bezglutenowa"]),
    workflow=WorkflowMemory(target_restaurant="Włoska Robota", target_date="Piątek", target_time="19:00", guest_count=4),
    working=WorkingMemory()
)

# Scenariusz krok po kroku
run_simulation_step(current_state, "Restauracja Włoska Robota, słucham?")
run_simulation_step(current_state, "Tak, zgadza się. W czym mogę pomóc?")
run_simulation_step(current_state, "Mamy wolny stolik o tej porze. Na jakie nazwisko rezerwacja?")
run_simulation_step(current_state, "Dobrze Pani Anno, zapisane. Piątek 19:00, stolik bezglutenowy.")


──────────────────────────────────────────────────────────────────────
📞 RECEPCJA: Restauracja Włoska Robota, słucham?

⚙️  KONTEKST:
ROLA: Jesteś osobistym asystentem dzwoniącym do restauracji w imieniu klienta.
TWOJA SZEFOWA (KLIENTKA): Anna Nowak
CEL: Zarezerwuj stolik w "Włoska Robota".

ZASADY:
- Nie podawaj danych karty kredytowej przez telefon.
- Jeśli nie ma miejsca o wybranej godzinie, zapytaj o +/- 30 minut.
- Bądź uprzejmy, ale realizuj cel.


AKTUALNY KROK: CallStep.INITIATION

INSTRUKCJE DLA TEGO KROKU (MECHANIKA KK):

    👉 Twoje zadanie: Przywitaj się i upewnij, że dodzwoniłeś się do "Włoska Robota".
    ⛔ ZAKAZ: Nie podawaj jeszcze daty ani liczby osób. Nie zasypuj rozmówcy informacjami.
    Przykład: "Dzień dobry, czy dodzwoniłem się do restauracji X?"

🤖 AGENT: Dzień dobry. Czy to restauracja Włoska Robota?

──────────────────────────────────────────────────────────────────────
📞 RECEPCJA: Tak, zgadza się. W czym mogę pomóc?

⚙️  KONTEKST:
ROLA: Jesteś osobistym asys

### 🧠 Analiza wyników: Co tu się właściwie stało?

Spójrz teraz na logi w konsoli. Sytuacja jest poprawna:

1. Tura 1 (*INITIATION*): Agent widzi instrukcję: "Upewnij się, że dodzwoniłeś się do Włoskiej Roboty". Nie widzi daty ani godziny. Pyta grzecznie o nazwę restauracji.
2. Tura 2 (*NEGOTIATION*): Dopiero gdy ma pewność, gdzie dzwoni, prompt ujawnia mu cel: "Piątek 19:00, 4 osoby". Ale nadal ukrywa nazwisko!
3. Tura 3 (*FINALIZATION*): Recepcja pyta o nazwisko. Dopiero teraz Agent "przypomina sobie" (dostaje w prompcie), że dzwoni w imieniu Anny Nowak i że ważna jest dieta bezglutenowa.

To jest właśnie inżynieria kontekstu z wykorzystaniem pamięci. Agent nie jest "wylewny". Działa jak profesjonalista, ujawniając informacje tylko wtedy, gdy są potrzebne w procesie.


## Bliźniacze Pary: Taxi Dispatcher powraca 🚕

Pamiętasz lekcję 1.4? To tam, w pocie czoła, budowaliśmy logikę dla **Taxi Dispatchera**. Definiowaliśmy stany (`TaxiBookingStatus`), tworzyliśmy model analizy (`TaxiResponseAnalysis`) i pisaliśmy pierwsze prompty systemowe.

Tamte elementy były **stateless** – działały świetnie w pojedynczej wymianie zdań. Ale teraz, w lekcji 2.2, weszliśmy na wyższy poziom. Nauczyliśmy się, że aby prowadzić dłuższą rozmowę i pamiętać kontekst (np. że klient to VIP albo że szukamy kierowcy już 10 minut), potrzebujemy struktury **Stateful**.

Twoim zadaniem jest teraz wzięcie tego "silnika", który zbudowaliśmy w lekcji 1.4, i włożenie go do "nadwozia" z pamięcią, które poznaliśmy dzisiaj.

> ☝️ **Ważna uwaga techniczna:**
> W pliku `helper.py` (z lekcji 1.4) masz już zdefiniowane:
> 1. `TaxiBookingStatus` (Enum)
> 2. `TaxiResponseAnalysis` (Model Pydantic)
> Nie musisz ich pisać od nowa. Skupimy się na obudowaniu ich warstwami pamięci: **Semantic** (Profil) oraz **Workflow** (Proces).

## 🧠 Zadanie 2.2.1: Definicja struktury pamięci (Taxi)

W przypadku taksówki "profil klienta" i "stan procesu" wyglądają inaczej niż w restauracji. Nie obchodzą nas alergie pokarmowe, ale metoda płatności – tak. Nie negocjujemy stolika, ale czekamy na przydział kierowcy.

Twoim zadaniem jest zaprojektowanie i utworzenie klasy `TaxiAgentState`, która będzie naszym "Jednym Źródłem Prawdy".

### Co dokładnie masz zrobić:
1.  Stwórz model `TaxiUserProfile` (Semantic Memory), który przechowuje: `user_id`, `preferred_payment` (np. Karta/Gotówka) oraz `default_pickup_address` (opcjonalnie)
2.  Stwórz model `TaxiWorkflowMemory`, który przechowuje: `status` (korzystając z `h.TaxiBookingStatus` z lekcji 1.4) oraz `assigned_car_details` (jako string lub osobny obiekt, opcjonalnie)
3.  Zepnij to w główną klasę `TaxiAgentState`

### Oczekiwany wynik:
Kod Python definiujący te klasy oraz utworzenie przykładowej instancji stanu dla klienta "Jan Kowalski", który woli płacić "Blikiem".


```json
{
  "user": {
    "name": "Jan Kowalski",
    "preferred_payment": "BLIK",
    "is_corporate_client": false
  },
  "workflow": {
    "status": "INITIATION",
    "assigned_car": null,
    "estimated_arrival_min": null
  }
}
```


<br />
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>
    
### Podpowiedź:    
- Skup się na hierarchii. `TaxiAgentState` ma zawierać w sobie instancje `user` i `workflow`.
- Pamiętaj o wartościach domyślnych (np. `status` na początku to zazwyczaj `INITIATION` lub `SEARCHING`).
- Użyj `Optional` dla pól, których na początku nie znamy (np. dane samochodu).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python
# 1. Semantic Memory (To co wiemy o kliencie)
class TaxiUserProfile(BaseModel):
    name: str
    preferred_payment: str = "Gotówka"
    is_corporate_client: bool = False

# 2. Workflow Memory (To co dzieje się w procesie)
class TaxiWorkflowMemory(BaseModel):
    # Wykorzystujemy Enum z lekcji 1.4
    status: h.TaxiBookingStatus = Field(default=h.TaxiBookingStatus.INITIATION)
    assigned_car: Optional[str] = None
    estimated_arrival_min: Optional[int] = None

# 3. Agregat (State Object)
class TaxiAgentState(BaseModel):
    user: TaxiUserProfile
    workflow: TaxiWorkflowMemory = Field(default_factory=TaxiWorkflowMemory)

# Test instancji
state = TaxiAgentState(
    user=TaxiUserProfile(name="Jan Kowalski", preferred_payment="BLIK")
)
print(state.model_dump_json(indent=2))
```

</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 2.2.2: Hard Control w Jinja2 (Context Engineering)

Masz już stan. Teraz zastosujemy mechanikę **KK (Końskie Klapki)**.
W procesie zamawiania taxi jest moment krytyczny: gdy szukamy kierowcy, agent **nie może** zmyślać, że ktoś już jedzie. Musi czekać.

Twoim zadaniem jest napisanie szablonu Jinja2, który dynamicznie ukrywa lub pokazuje informacje w zależności od statusu w `TaxiWorkflowMemory`.

### Oczekiwany wynik:
Dwa stringi. Jeden "ubogi" w informacje (tylko prośba o cierpliwość), drugi "bogaty" (z konkretnym autem). Model w pierwszym przypadku fizycznie nie będzie mógł zhalucynować samochodu.


### Przykładowy wynik

```txt
--- ITERATION 1 ---

Jesteś dyspozytorem taxi.

STATUS: AWAITING_DRIVER_ASSIGNMENT

Instrukcje:

- Twoim jedynym celem jest utrzymanie klienta na linii.
- Powiedz: "Szukam kierowcy w Twojej okolicy, proszę o chwilę cierpliwości".
- NIE PODAWAJ żadnych szczegółów pojazdu, bo ich jeszcze nie masz.


--- ITERATION 2 ---

Jesteś dyspozytorem taxi.

STATUS: DRIVER_ASSIGNED

Instrukcje:

- Kierowca znaleziony!
- Przekaż: "Podjeżdża Srebrna Toyota Prius (WX 9977)".
- Życz miłej podróży.

```

<br />
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>
    
### Podpowiedź: 
    
Odwołuj się do struktury z zadania wyżej: `state.workflow.status`.
Pamiętaj, że w Jinja2 dostęp do wartości Enuma to często po prostu nazwa (jako string) lub `.value`.
Dla testu stwórz obiekt stanu, zmień mu ręcznie status i wyrenderuj szablon. Potem zmień status znowu i wyrenderuj ponownie.
    
    
### Co dokładnie masz zrobić:
1.  Stwórz szablon promptu (`taxi_template`).
2.  Zaimplementuj logikę `{% if %}`:
    *   Gdy status to `AWAITING_DRIVER_ASSIGNMENT`: Agent ma instrukcję: "Poproś klienta o cierpliwość, szukamy kierowcy." (Nie widzi danych auta).
    *   Gdy status to `DRIVER_ASSIGNED`: Agent widzi dane auta (wstrzyknięte ze zmiennej) i ma instrukcję: "Potwierdź, że jedzie [Auto X]."
3.  Wyrenderuj szablon dla obu stanów, aby udowodnić, że prompt się zmienia.
    

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python

template_str = """
Jesteś dyspozytorem taxi.

STATUS: {{ state.workflow.status.value }}

Instrukcje:
{% if state.workflow.status.value == 'AWAITING_DRIVER_ASSIGNMENT' %}
- Twoim jedynym celem jest utrzymanie klienta na linii.
- Powiedz: "Szukam kierowcy w Twojej okolicy, proszę o chwilę cierpliwości".
- NIE PODAWAJ żadnych szczegółów pojazdu, bo ich jeszcze nie masz.
{% elif state.workflow.status.value == 'DRIVER_ASSIGNED' %}
- Kierowca znaleziony!
- Przekaż: "Podjeżdża {{ state.workflow.assigned_car }}".
- Życz miłej podróży.
{% endif %}
"""

t = Template(template_str)

# Przypadek 1: Szukamy
state.workflow.status = h.TaxiBookingStatus.AWAITING_DRIVER_ASSIGNMENT
print("--- ITERATION 1 ---")
print(t.render(state=state))

# Przypadek 2: Znalazł się
state.workflow.status = h.TaxiBookingStatus.DRIVER_ASSIGNED
state.workflow.assigned_car = "Srebrna Toyota Prius (WX 9977)"
print("\n--- ITERATION 2 ---")
print(t.render(state=state))
```

</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 2.2.3**: Pełna symulacja "The Loop"

To zadanie oznaczone**dwoma gwiazdkami**, bo wymaga połączenia wszystkich koncepcji: pamięć + kontekst) +  logika sterująca.

Twoim celem jest stworzenie **kompletnej symulacji pętli agenta taksówkowego**, dokładnie tak, jak zrobiliśmy to dla restauracji. Nie będziemy tu dzwonić do prawdziwego API OpenAI – zamiast tego **symulujemy** działanie mózgu agenta, pisząc logikę aktualizacji stanu w Pythonie.

Chcemy zobaczyć, jak Twój system reaguje na zmieniającą się rozmowę i jak "Końskie Klapki" w szablonie zmieniają instrukcje w zależności od etapu.

### Co dokładnie masz zrobić:

1.  **Zainicjuj stan (`TaxiAgentState`):**
    *   Klient: "Tomasz Igrekowski", płatność "Karta Firmowa".
    *   Workflow: Cel podróży "Dworzec Centralny", Status początkowy `INITIATION`.
2.  **Stwórz szablon Jinja2:**
    *   Musi obsługiwać stany: `INITIATION`, `SEARCHING` (szukanie kierowcy) oraz `DRIVER_ASSIGNED` (kierowca znaleziony).
    *   Użyj `{% if %}`, aby ukrywać/pokazywać instrukcje (np. pokaż polecenie "Podaj dane auta" TYLKO w stanie `DRIVER_ASSIGNED`).
3.  **Napisz funkcję `run_taxi_simulation_step(state, dispatcher_voice)`:**
    *   Funkcja przyjmuje obecny stan i głos dyspozytora.
    *   **Krok 1 (Input):** Dodaje głos dyspozytora do historii (`WorkingMemory`).
    *   **Krok 2 (Render):** Renderuje szablon Jinja2 na podstawie obecnego stanu (to symuluje to, co "widzi" model). Wyprintuj to na konsoli jako "MÓZG AGENTA".
    *   **Krok 3 (Logic & Update):** Napisz prostą logikę `if/else`, która aktualizuje stan:
        *   Jeśli dyspozytor zapyta "Dokąd?", przejdź do `SEARCHING`.
        *   Jeśli dyspozytor powie "Mamy kierowcę" lub poda markę auta, przejdź do `DRIVER_ASSIGNED` i zapisz dane auta w pamięci.
        * Pamiętaj, aby obsłużyć również sytuację braku dopasowania (else), aby agent nie milczał, gdy nie rozpozna słowa kluczowego.
    *   **Krok 4 (Output):** Wygeneruj (na sztywno lub prostym printem) odpowiedź agenta i dodaj ją do historii.
4.  **Uruchom scenariusz:** Przeprowadź rozmowę składającą się z 3-4 tur, która doprowadzi do sukcesu.

### Oczekiwany wynik końcowy:
Kod, który po uruchomieniu wyświetli logi z całej rozmowy, pokazując jak zmienia się "MÓZG AGENTA" (prompt) i jak aktualizuje się obiekt stanu w pamięci.

### Przykładowy wynik
```txt
📞 DYSPOZYTOR: Taxi Halo, słucham?

--- MÓZG AGENTA (PROMPT) ---
Jesteś asystentem zamawiającym taxi dla: Tomasz Igrekowski.
Metoda płatności: Karta Firmowa.
Cel: Dworzec Centralny.

AKTUALNY STATUS: INITIATION

INSTRUKCJE:

- Przywitaj się i powiedz, że chcesz zamówić kurs.

----------------------------
🤖 AGENT: Dzień dobry, poproszę kurs na Dworzec Centralny.

📞 DYSPOZYTOR: Dobrze, szukam kierowcy w okolicy...

--- MÓZG AGENTA (PROMPT) ---
Jesteś asystentem zamawiającym taxi dla: Tomasz Igrekowski.
Metoda płatności: Karta Firmowa.
Cel: Dworzec Centralny.

AKTUALNY STATUS: SEARCHING

INSTRUKCJE:

- Podałeś cel. Czekaj na informację o kierowcy.
- Jeśli dyspozytor każe czekać -> Potwierdź, że czekasz.
- NIE PODAWAJ ŻADNYCH DANYCH AUTA - jeszcze ich nie znasz!

----------------------------
🤖 AGENT: Rozumiem, czekam na informację.

📞 DYSPOZYTOR: Mam dla Pana czarną Toyotę, będzie za 5 minut.

--- MÓZG AGENTA (PROMPT) ---
Jesteś asystentem zamawiającym taxi dla: Tomasz Igrekowski.
Metoda płatności: Karta Firmowa.
Cel: Dworzec Centralny.

AKTUALNY STATUS: SEARCHING

INSTRUKCJE:

- Podałeś cel. Czekaj na informację o kierowcy.
- Jeśli dyspozytor każe czekać -> Potwierdź, że czekasz.
- NIE PODAWAJ ŻADNYCH DANYCH AUTA - jeszcze ich nie znasz!

----------------------------
🤖 AGENT: Rozumiem, czekam na informację.
TaxiAgentState(user=TaxiUserProfile(name='Tomasz Igrekowski', payment_method='Karta Firmowa'), workflow=TaxiWorkflowMemory(status=<TaxiBookingStatus.SEARCHING: 'SEARCHING'>, destination='Dworzec Centralny', assigned_car=None), working=WorkingMemory(history=['Dispatcher: Taxi Halo, słucham?', 'Agent: Dzień dobry, poproszę kurs na Dworzec Centralny.', 'Dispatcher: Dobrze, szukam kierowcy w okolicy...', 'Agent: Rozumiem, czekam na informację.', 'Dispatcher: Mam dla Pana czarną Toyotę, będzie za 5 minut.', 'Agent: Rozumiem, czekam na informację.']))
```


<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>
    
### Podpowiedź:    
To zadanie wydaje się duże, ale składa się z trzech bloków.

1. **Szablon:** Skup się na warunkach.
   ```jinja2
   {% if state.workflow.status == 'SEARCHING' %}
      Instrukcja: Czekaj cierpliwie. Nie masz jeszcze danych auta.
   {% elif state.workflow.status == 'DRIVER_ASSIGNED' %}
      Instrukcja: Potwierdź przejazd autem: {{ state.workflow.assigned_car }}.
   {% endif %}
   ```

2. **Symulacja Logiki:**
   Wewnątrz funkcji `run_simulation` musisz "udawać" wnioskowanie modelu.
   ```python
   # Mockowanie decyzji AI
   if "dokąd" in dispatcher_voice.lower():
       state.workflow.status = TaxiBookingStatus.SEARCHING
       response = "Na Dworzec Centralny."
   elif "toyota" in dispatcher_voice.lower():
       state.workflow.status = TaxiBookingStatus.DRIVER_ASSIGNED
       state.workflow.assigned_car = "Toyota" # Ekstrakcja danych
       response = "Dziękuję, czekam."
   ```

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈 </summary>
<p>

```python

# --- 1. Definicje Struktur (pamięć) ---

class TaxiBookingStatus(str, Enum):
    INITIATION = "INITIATION"
    SEARCHING = "SEARCHING"
    DRIVER_ASSIGNED = "DRIVER_ASSIGNED"

class TaxiUserProfile(BaseModel):
    name: str
    payment_method: str

class TaxiWorkflowMemory(BaseModel):
    status: TaxiBookingStatus = TaxiBookingStatus.INITIATION
    destination: str
    assigned_car: Optional[str] = None

class WorkingMemory(BaseModel):
    history: List[str] = Field(default_factory=list)

class TaxiAgentState(BaseModel):
    user: TaxiUserProfile
    workflow: TaxiWorkflowMemory
    working: WorkingMemory = Field(default_factory=WorkingMemory)

# --- 2. Inżynieria Kontekstu (szablon) ---

taxi_template_str = """
--- MÓZG AGENTA (PROMPT) ---
Jesteś asystentem zamawiającym taxi dla: {{ state.user.name }}.
Metoda płatności: {{ state.user.payment_method }}.
Cel: {{ state.workflow.destination }}.

AKTUALNY STATUS: {{ state.workflow.status.value }}

INSTRUKCJE:
{% if state.workflow.status.value == 'INITIATION' %}
- Przywitaj się i powiedz, że chcesz zamówić kurs.
{% elif state.workflow.status.value == 'SEARCHING' %}
- Podałeś cel. Czekaj na informację o kierowcy.
- Jeśli dyspozytor każe czekać -> Potwierdź, że czekasz.
- NIE PODAWAJ ŻADNYCH DANYCH AUTA - jeszcze ich nie znasz!
{% elif state.workflow.status.value == 'DRIVER_ASSIGNED' %}
- Mamy sukces! Kierowca jedzie.
- Potwierdź przyjazd auta: {{ state.workflow.assigned_car }}.
{% endif %}
----------------------------
"""

# --- 3. Silnik symulacji (The Loop) ---

def run_taxi_simulation_step(state: TaxiAgentState, dispatcher_voice: str):
    # A. Input
    print(f"\n📞 DYSPOZYTOR: {dispatcher_voice}")
    state.working.history.append(f"Dispatcher: {dispatcher_voice}")
    
    # B. Renderowanie (Co widzi agent?)
    t = Template(taxi_template_str)
    rendered_prompt = t.render(state=state)
    print(rendered_prompt)
    
    # C. Mockowanie Logiki (To normalnie robiłby LLM + Structured Output)
    ai_response = ""
    
    # Logika dla stanu INITIATION
    if state.workflow.status == TaxiBookingStatus.INITIATION:
        if "dokąd" in dispatcher_voice.lower() or "słucham" in dispatcher_voice.lower():
            state.workflow.status = TaxiBookingStatus.SEARCHING
            ai_response = f"Dzień dobry, poproszę kurs na {state.workflow.destination}."
            
    # Logika dla stanu SEARCHING
    elif state.workflow.status == TaxiBookingStatus.SEARCHING:
        if "toyota" in dispatcher_voice.lower():
            # TRIGGER: Znaleziono auto -> Zmiana stanu
            state.workflow.status = TaxiBookingStatus.DRIVER_ASSIGNED
            state.workflow.assigned_car = "Czarna Toyota (WA 12345)" 
            ai_response = "Super, potwierdzam Toyotę. Czekam."
        else:
            # HOLD: Brak trigera -> Czekamy (Tego brakowało!)
            ai_response = "Rozumiem, czekam na informację."
            
    # Logika dla stanu DRIVER_ASSIGNED
    elif state.workflow.status == TaxiBookingStatus.DRIVER_ASSIGNED:
        ai_response = "Dziękuję, do widzenia."

    # D. Output
    print(f"🤖 AGENT: {ai_response}")
    state.working.history.append(f"Agent: {ai_response}")
    return state

# --- 4. Uruchomienie scenariusza ---

# Inicjalizacja
current_state = TaxiAgentState(
    user=TaxiUserProfile(name="Tomasz Igrekowski", payment_method="Karta Firmowa"),
    workflow=TaxiWorkflowMemory(destination="Dworzec Centralny")
)

# Tura 1: Start -> Przejście do SEARCHING
run_taxi_simulation_step(current_state, "Taxi Halo, słucham?")

# Tura 2: Negocjacja -> Pozostanie w SEARCHING 
run_taxi_simulation_step(current_state, "Dobrze, szukam kierowcy w okolicy...")

# Tura 3: Znalezienie -> Przejście do DRIVER_ASSIGNED
run_taxi_simulation_step(current_state, "Mam dla Pana czarną Toyotę, będzie za 5 minut.")
```

</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## Podsumowanie

W tej lekcji przestaliśmy "gadać o pamięci", a zaczęliśmy ją "budować". Przeszliśmy drogę od luźnych koncepcji do działającego kodu.

Oto najważniejsze wnioski inżynierskie, które zabierasz z tej lekcji:

1.  **Pamięć to konkretny kod:** Zrozumiałeś, że w Software 3.0 pamięć to nie abstrakcyjna chmura, ale precyzyjnie otypowane klasy Pydantic. Jeśli czegoś nie ma w kodzie, agent tego nie pamięta.
2.  **Architektura Stateful:** Oddzieliliśmy dane (Memory Object) od logiki (Python) i prezentacji (Jinja). Dzięki temu system jest "stateful" (pamięta stan), mimo że sam model LLM jest "stateless".
3.  **Sterowanie przez ukrywanie:** Nauczyłeś się, że najlepszą kontrolą nad modelem nie jest proszenie go ("proszę, nie mów tego"), ale fizyczne odcięcie dostępu do informacji za pomocą `Workflow Memory` i warunków w Jinja2. To, czego model nie widzi, nie może go rozproszyć.
4.  **Odpowiedzialność warstw:** Wiesz już, że `Core Memory` jest nienaruszalne (ReadOnly), a `Working Memory` to tylko ulotne światło latarki. Pomylenie tych warstw (np. trzymanie stałych preferencji w ulotnej historii) to prosty przepis na "amnezję" agenta.
5.  **Bez bazy danych też działa:** Udowodniliśmy, że można zbudować w pełni funkcjonalny, złożony system pamięci używając wyłącznie obiektów w Pythonie. Baza danych jest mechanizmem trwałości (też np. skalowalności), a nie logiki.

### Warstwy pamięci w pigułce

Poniższa tabela to Twoja ściągawka inżynierska. Pomoże Ci szybko zdecydować, gdzie dany fragment informacji powinien trafić i jak nim zarządzać.

| Cecha | 🏛️ Core (Constitutional) | 🧠 Semantic (Knowledge) | 🎯 Workflow (Process) | ⚡ Working (Context) |
| :--- | :--- | :--- | :--- | :--- |
| **Odpowiedzialność** | Tożsamość, bezpieczeństwo, DNA agenta. | Długotrwała wiedza o użytkowniku i świecie. | Stan procesu biznesowego (gdzie jesteśmy?). | Pamięć operacyjna, kontekst "tu i teraz". |
| **Przykłady** | "Nie pytaj o PIN", "Bądź uprzejmy", "Max rabat 10%". | "Imię: Anna", "Status: VIP", "Alergia: Orzechy". | `step: PAYMENT`, `attempts: 2`, `data: {date: "2024-11-20"}`. | Ostatnie 10 wiadomości, "brudnopis" myśli, błędy API. |
| **Mutowalność** | **READ-ONLY** (Immutable) | **READ-WRITE** (Persistent) | **READ-WRITE** (Volatile) | **APPEND-ONLY** / **SLIDING** |
| **Trigger zmiany** | **Deployment** (zmiana w kodzie/konfigu). | **Ekstrakcja** (gdy pojawi się nowy fakt) | **Każda tura** (step by step). | **Ciągły** (strumień zdarzeń). |
| **Cykl życia** | Od startu do stopu aplikacji. | Dni, miesiące, lata (jak CRM) | Czas trwania jednej transakcji/rozmowy. | Czas trwania sesji (ulotna). |
| **Bramka jakości** | Code Review & Tests (człowiek). | Weryfikacja faktów (model + reguły). | Strict Schema (Pydantic/Enum) | Brak (surowe logi) |
| **Gdy brakuje...** | Agent jest niebezpieczny, "loose cannon". | Agent ma "amnezję", nie poznaje stałych klientów. | Agent gubi wątek, zapętla się, nie dowozi celu | Agent nie rozumie kontekstu "tu i teraz" ("zmień *to*") |

---

## Co dalej?

W tej lekcji pracowaliśmy na "żywym organizmie" w jednym notatniku, żeby widzieć wszystkie zależności w jednym miejscu. To świetne do nauki, ale w systemach produkcyjnych potrzebujemy lepszej organizacji.

Dlatego w kolejnym kroku zrobimy **wielkie inżynierskie porządki**. Poukładamy te wszystkie elementy (modele Pydantic, helpery, szablony) w profesjonalną strukturę folderów i plików.

A gdy fundamenty będą już czyste i solidne, rozpoczniemy pracę nad zupełnie nowym wyzwaniem – use case'em **uczenia języka obcego**, gdzie techniki pamięciowe wejdą na jeszcze wyższy poziom złożoności.

## Przydatne linki
1. [Simon Willison’s Weblog - context engineering](https://simonwillison.net/2025/Jun/27/context-engineering/)
2. [The New Skill in AI is Not Prompting, It's Context Engineering](https://www.philschmid.de/context-engineering)
3. [How we built our multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system)
4. [Why Multi-Agent Systems Need Memory Engineering](https://www.mongodb.com/company/blog/technical/why-multi-agent-systems-need-memory-engineering)
5. [Why Do Multi-Agent LLM Systems Fail?](https://arxiv.org/pdf/2503.13657)
6. [Memory Optimization Strategies in AI Agents](https://diamantai.substack.com/p/memory-optimization-strategies-in)
7. [Beyond the Prototype: 15 Hard-Earned Lessons to Ship Production-Ready AI Agents](https://medium.com/data-science-collective/beyond-the-prototype-15-hard-earned-lessons-to-ship-production-ready-ai-agents-e58139d80299)